# WAN 2.2 — Geração e Extensão de Vídeo a partir de Imagem

Prova de conceito usando o pacote oficial **Wan2.2** (MoE A14B fp8) com os modelos do Comfy-Org Repackaged.

### Arquitetura MoE A14B
- **high_noise_model** — expert de alta frequência de ruído: define o layout geral nos primeiros passos do denoising.
- **low_noise_model** — expert de baixo ruído: refina detalhes e qualidade nos passos finais.
- A transição entre os experts é controlada por `boundary` (padrão `0.9`). O modelo usa ~14B parâmetros ativos por passo (total de ~27B).
- **LoRA LightX2V 4-steps**: acelera a geração para apenas 4 passos de sampling (Lightning).

### Hardware recomendado
- NVIDIA Ampere / Ada Lovelace / Hopper (24 GB+ VRAM para `offload_model=True`).
- `offload_model=True` descarrega modelos para CPU entre timesteps, economizando VRAM.

**Referências:**
- [WAN 2.2 GitHub](https://github.com/Wan-Video/Wan2.2)
- [Comfy-Org WAN 2.2 Repackaged (HuggingFace)](https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged)
- [Wan-AI/Wan2.2-I2V-A14B (HuggingFace)](https://huggingface.co/Wan-AI/Wan2.2-I2V-A14B)

## 1. Instalação das Dependências

Instala o pacote oficial `wan` diretamente do GitHub e todas as dependências necessárias.
A instalação do `flash-attn` é opcional — se falhar, o PyTorch usa o backend `sdpa` automaticamente.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Instalando PyTorch (CUDA 12.1)...")
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("Instalando dependências base...")
%pip install --quiet \
    "diffusers>=0.33.0" \
    "transformers>=4.45.0" \
    "accelerate>=0.33.0" \
    "opencv-python>=4.10.0" \
    "imageio>=2.35.0" \
    "imageio-ffmpeg>=0.5.1" \
    "Pillow>=10.0.0" \
    "huggingface_hub>=0.24.0" \
    "hf-transfer>=0.1.8" \
    "safetensors>=0.4.3" \
    "sentencepiece" \
    "protobuf" \
    "easydict"

print("Instalando flash-attn...")
%pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.6cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

print("Instalando pacote oficial wan (Wan2.2)...")
%pip install --quiet "git+https://github.com/Wan-Video/Wan2.2.git" --no-deps

print("\n✅ Instalação concluída. Reinicie o kernel se necessário.")

## 2. Importação e Verificação

In [ ]:
import gc
import os
import sys
from pathlib import Path

import imageio
import numpy as np
import torch
from diffusers.utils import export_to_video, load_image
from IPython.display import Video, display
from PIL import Image

# --- Pacote oficial wan ---
try:
    from wan.image2video import WanI2V
    from wan.configs import WAN_CONFIGS
    print("✅ Pacote 'wan' (Wan2.2) carregado com sucesso.")
except ImportError as e:
    raise ImportError(
        "❌ Pacote 'wan' não encontrado. "
        "Execute a célula de instalação e reinicie o kernel.\n" + str(e)
    )

# --- Flash Attention (opcional) ---
try:
    import flash_attn
    print(f"⚡ Flash Attention {flash_attn.__version__} disponível.")
except ImportError:
    print("⚠️  Flash Attention não disponível — usando backend sdpa do PyTorch.")

# --- GPU ---
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU não disponível — execução em CPU será muito lenta.")

# --- Chaves disponíveis no WAN_CONFIGS ---
print(f"\nModelos disponíveis em WAN_CONFIGS: {list(WAN_CONFIGS.keys())}")

## 3. Download dos Modelos (Comfy-Org Repackaged)

Estrutura de diretório esperada pelo `WanI2V` após o download:

```
models/wan22/
├── low_noise_model/          ← subfolder para WanModel.from_pretrained()
│   └── model.safetensors     ← symlink → wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors
├── high_noise_model/         ← subfolder para WanModel.from_pretrained()
│   └── model.safetensors     ← symlink → wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors
├── models/
│   └── t5_encoder/           ← t5_checkpoint do config
│       └── model.safetensors ← symlink → umt5_xxl_fp8_e4m3fn_scaled.safetensors
├── Wan2.1_VAE.pth            ← vae_checkpoint do config (nome exato)
├── wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors
├── wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors
├── wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors
├── wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors
└── umt5_xxl_fp8_e4m3fn_scaled.safetensors
```

> **Nota sobre nomes:** O config `i2v-A14B` espera `vae_checkpoint = 'Wan2.1_VAE.pth'`.
> O arquivo HF (`wan_2.1_vae.safetensors`) é renomeado/linkado como `Wan2.1_VAE.pth` por compatibilidade.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID   = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"
MODELS_DIR   = Path("models/wan22")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Mapeamento: nome_local → caminho no repositório HF
HF_FILES = {
    "wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors"   : "split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors",
    "wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors"    : "split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors",
    "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors"  : "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",
    "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors" : "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors",
    "umt5_xxl_fp8_e4m3fn_scaled.safetensors"             : "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "wan_2.1_vae.safetensors"                            : "split_files/vae/wan_2.1_vae.safetensors",
}

def download_models(repo_id: str, models_dir: Path, hf_files: dict) -> dict:
    """Baixa arquivos do HF Hub para models_dir, pulando os já existentes."""
    local_paths = {}
    for local_name, hf_path in hf_files.items():
        dest = models_dir / local_name
        if dest.exists():
            print(f"  ✓ {local_name} (já existe)")
        else:
            print(f"  ⬇️  Baixando {local_name}...")
            hf_hub_download(
                repo_id=repo_id,
                filename=hf_path,
                local_dir=str(models_dir),
                local_dir_use_symlinks=False,
            )
            # Move para o nome local esperado (sem subpastas do HF)
            downloaded = models_dir / hf_path
            if downloaded.exists() and not dest.exists():
                downloaded.rename(dest)
        local_paths[local_name] = dest
    return local_paths


def setup_checkpoint_dir(models_dir: Path, local_paths: dict):
    """
    Cria a estrutura de subdiretórios e symlinks que WanI2V.from_pretrained() espera.

    WanModel.from_pretrained(checkpoint_dir, subfolder='high_noise_model') procura:
        checkpoint_dir/high_noise_model/model.safetensors  (ou config.json + shards)

    T5EncoderModel espera:
        checkpoint_dir/<t5_checkpoint>  →  models/t5_encoder/

    VAE (Wan2_1_VAE) espera:
        checkpoint_dir/Wan2.1_VAE.pth   (nome exato do config)
    """
    def _make_symlink(link_path: Path, target: Path):
        """Cria (ou recria) um symlink relativo."""
        if link_path.is_symlink() or link_path.exists():
            link_path.unlink()
        # symlink relativo ao diretório pai do link
        rel_target = os.path.relpath(target, link_path.parent)
        link_path.symlink_to(rel_target)

    # --- high_noise_model/ ---
    high_dir = models_dir / "high_noise_model"
    high_dir.mkdir(exist_ok=True)
    _make_symlink(
        high_dir / "model.safetensors",
        local_paths["wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors"]
    )

    # --- low_noise_model/ ---
    low_dir = models_dir / "low_noise_model"
    low_dir.mkdir(exist_ok=True)
    _make_symlink(
        low_dir / "model.safetensors",
        local_paths["wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors"]
    )

    # --- T5 encoder: models/t5_encoder/model.safetensors ---
    t5_dir = models_dir / "models" / "t5_encoder"
    t5_dir.mkdir(parents=True, exist_ok=True)
    _make_symlink(
        t5_dir / "model.safetensors",
        local_paths["umt5_xxl_fp8_e4m3fn_scaled.safetensors"]
    )

    # --- VAE: Wan2.1_VAE.pth (nome exato esperado pelo config) ---
    _make_symlink(
        models_dir / "Wan2.1_VAE.pth",
        local_paths["wan_2.1_vae.safetensors"]
    )

    print("✅ Estrutura de checkpoint configurada.")


print("📦 Verificando / baixando modelos...")
local_paths = download_models(HF_REPO_ID, MODELS_DIR, HF_FILES)
setup_checkpoint_dir(MODELS_DIR, local_paths)
print("\nArquivos em MODELS_DIR:")
for p in sorted(MODELS_DIR.rglob("*")):
    print(" ", p.relative_to(MODELS_DIR), "→ symlink" if p.is_symlink() else "")

## 4. Inicialização do Pipeline WanI2V

O `WanI2V` carrega:
1. **T5EncoderModel** — codificador de texto (fica na CPU com `t5_cpu=True`).
2. **Wan2_1_VAE** — encoder/decoder de vídeo.
3. **WanModel (high_noise)** — expert MoE para os primeiros timesteps (`t >= boundary * num_train_timesteps`).
4. **WanModel (low_noise)** — expert MoE para os timesteps finais.

Os **LoRAs LightX2V** são carregados separadamente e permitem gerar em **4 passos** com qualidade comparável a 40 passos nativos.

In [ ]:
# --- Escolha da chave de config ---
# WAN_CONFIGS pode ter 'i2v-A14B' ou 'i2v-14B' dependendo da versão instalada.
# Detectamos automaticamente:
_CONFIG_KEY = None
for key in ("i2v-A14B", "i2v-14B"):
    if key in WAN_CONFIGS:
        _CONFIG_KEY = key
        break
if _CONFIG_KEY is None:
    raise KeyError(
        f"Nenhuma chave I2V encontrada em WAN_CONFIGS. "
        f"Chaves disponíveis: {list(WAN_CONFIGS.keys())}"
    )
print(f"Usando config: '{_CONFIG_KEY}'")


def load_wan_i2v(
    checkpoint_dir: Path,
    lora_paths: dict,
    config_key: str = _CONFIG_KEY,
    device_id: int = 0,
) -> WanI2V:
    """
    Instancia WanI2V com os modelos high/low noise e aplica os LoRAs LightX2V.

    Parâmetros
    ----------
    checkpoint_dir : Path
        Diretório raiz que contém as subpastas high_noise_model/, low_noise_model/,
        models/t5_encoder/ e o arquivo Wan2.1_VAE.pth.
    lora_paths : dict
        Dicionário com as chaves 'lora_high' e 'lora_low' apontando para os
        arquivos .safetensors dos LoRAs LightX2V.
    config_key : str
        Chave do modelo em WAN_CONFIGS (e.g. 'i2v-A14B').
    device_id : int
        ID da GPU a ser usada.
    """
    cfg = WAN_CONFIGS[config_key]

    # boundary=0.9 → primeiros 90% dos timesteps usam high_noise, últimos 10% usam low_noise
    # (valor padrão do config; ajuste para 0.875 se quiser mais refinamento)
    cfg.boundary = 0.900

    print(f"Carregando WanI2V (config='{config_key}', boundary={cfg.boundary})...")
    wan_i2v = WanI2V(
        config=cfg,
        checkpoint_dir=str(checkpoint_dir),
        device_id=device_id,
        t5_cpu=True,        # mantém T5 na CPU — economiza VRAM
        init_on_cpu=True,   # inicializa DiT na CPU e move para GPU por demanda
    )
    print("  ✅ Modelos base carregados.")

    # --- Aplicação dos LoRAs LightX2V ---
    # O WanModel expõe load_lora() somente em certas versões do pacote.
    # Verificamos antes de chamar.
    for model_attr, lora_key in [
        ("high_noise_model", "lora_high"),
        ("low_noise_model",  "lora_low"),
    ]:
        model = getattr(wan_i2v, model_attr)
        lora_file = lora_paths.get(lora_key)
        if lora_file and lora_file.exists():
            if hasattr(model, "load_lora"):
                print(f"  ⚡ Aplicando LoRA LightX2V em {model_attr}...")
                model.load_lora(str(lora_file))
            else:
                print(
                    f"  ⚠️  {model_attr}.load_lora() não disponível nesta versão do pacote. "
                    "O LoRA não será aplicado — use 40 passos para qualidade equivalente."
                )
        else:
            print(f"  ⚠️  LoRA '{lora_key}' não encontrado em {lora_file}.")

    return wan_i2v


lora_paths = {
    "lora_high": local_paths["wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors"],
    "lora_low":  local_paths["wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors"],
}

pipe = load_wan_i2v(MODELS_DIR, lora_paths)
print("\n🚀 Pipeline pronto para geração.")

## 5. Configuração de Geração

### Parâmetros principais do `WanI2V.generate()`

| Parâmetro | Recomendado (Lightning) | Nativo (sem LoRA) | Descrição |
|---|---|---|---|
| `sampling_steps` | **4** | 40 | Passos de denoising |
| `shift` | **5.0** | 5.0 | Sigma shift (3.0 para 480p, 5.0 para 720p) |
| `guide_scale` | **5.0** | 5.0 | CFG scale — pode ser `float` ou `(low, high)` |
| `frame_num` | **81** | 81 | Frames (deve ser 4n+1) |
| `max_area` | 480×832 | 720×1280 | Resolução máxima em pixels |
| `offload_model` | **True** | True | Offload para CPU entre timesteps |

> **`guide_scale` como tupla:** passe `(guide_scale_low, guide_scale_high)` para controlar
> independentemente o CFG de cada expert. Ex.: `(3.0, 7.0)`.

In [ ]:
# ── Imagem de entrada ─────────────────────────────────────────────────────────
INPUT_IMAGE_PATH = (
    "https://huggingface.co/datasets/huggingface/documentation-images"
    "/resolve/main/diffusers/astronaut.jpg"
)

# ── Resolução ─────────────────────────────────────────────────────────────────
# 480p → MAX_AREA = 480*832 = 399_360   |  shift = 3.0
# 720p → MAX_AREA = 720*1280 = 921_600  |  shift = 5.0
MAX_AREA = 480 * 832   # 480p — adequado para GPUs com 24 GB
SHIFT    = 3.0         # recomendado para 480p

# ── Sampling ──────────────────────────────────────────────────────────────────
SAMPLING_STEPS  = 4        # 4 com LoRA Lightning | 40 sem LoRA
GUIDE_SCALE     = 5.0      # float ou (low_cfg, high_cfg)
FRAME_NUM       = 81       # deve ser 4n+1
FPS             = 16

# ── Extensão iterativa ────────────────────────────────────────────────────────
EXTENSION_LOOPS = 2        # quantas vezes estender o vídeo
OVERLAP_FRAMES  = 1        # frames removidos do início de cada extensão na concatenação

# ── Prompt ────────────────────────────────────────────────────────────────────
PROMPT = (
    "An astronaut walks slowly across a barren alien landscape, "
    "cinematic lighting, smooth camera movement."
)
NEG_PROMPT = ""  # string vazia → usa config.sample_neg_prompt do modelo

# ── Saída ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs/wan_poc")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuração carregada.")
print(f"  Resolução máxima : {MAX_AREA} px² (~{int((MAX_AREA**0.5))}x{int((MAX_AREA**0.5))} equiv.)")
print(f"  Passos de sampling: {SAMPLING_STEPS}")
print(f"  Frames por segmento: {FRAME_NUM} @ {FPS} fps = {FRAME_NUM/FPS:.1f}s")
print(f"  Extensões: {EXTENSION_LOOPS}  →  vídeo final ≈ {(EXTENSION_LOOPS+1)*FRAME_NUM/FPS:.1f}s")

## 6. Geração e Extensão Iterativa

Fluxo:
1. **Segmento inicial** — gerado a partir da imagem de entrada.
2. **Extensões** — o último frame de cada segmento torna-se a imagem de entrada do próximo,
   mantendo continuidade visual.
3. **Concatenação** — todos os segmentos são unidos em um único vídeo final.

In [ ]:
def preprocess_image(image_path: str, max_area: int) -> Image.Image:
    """
    Carrega e redimensiona a imagem para que sua área não exceda `max_area`,
    mantendo o aspect ratio e alinhando dimensões a múltiplos de 16.
    """
    image = load_image(image_path)
    w, h = image.size
    aspect_ratio = h / w
    new_w = int((max_area / aspect_ratio) ** 0.5) // 16 * 16
    new_h = int((max_area * aspect_ratio) ** 0.5) // 16 * 16
    return image.resize((new_w, new_h), Image.LANCZOS)


def generate_segment(
    wan_i2v: WanI2V,
    image: Image.Image,
    prompt: str,
    neg_prompt: str = "",
    seed: int = 42,
    sampling_steps: int = SAMPLING_STEPS,
    shift: float = SHIFT,
    guide_scale = GUIDE_SCALE,
    frame_num: int = FRAME_NUM,
    output_path: str = "output.mp4",
) -> list:
    """
    Gera um segmento de vídeo e salva em output_path.

    Retorna a lista de frames PIL.

    O tensor retornado por WanI2V.generate() tem shape (C, N, H, W) com
    valores normalizados em [-1, 1] (saída do VAE decoder).
    """
    w, h = image.size
    video_tensor = wan_i2v.generate(
        input_prompt=prompt,
        img=image,
        max_area=w * h,
        frame_num=frame_num,
        shift=shift,
        sample_solver="unipc",
        sampling_steps=sampling_steps,
        guide_scale=guide_scale,
        n_prompt=neg_prompt,
        seed=seed,
        offload_model=True,
    )  # shape: (C=3, N, H, W), dtype: float32, range: [-1, 1]

    # Converte para frames PIL: (N, H, W, C) em [0, 255]
    video_np = video_tensor.cpu().float().numpy()          # (C, N, H, W)
    video_np = video_np.transpose(1, 2, 3, 0)             # (N, H, W, C)
    video_np = np.clip((video_np + 1.0) / 2.0, 0, 1)      # [-1,1] → [0,1]
    video_np = (video_np * 255).round().astype(np.uint8)

    frames = [Image.fromarray(f) for f in video_np]
    export_to_video(frames, output_path, fps=FPS)

    del video_tensor, video_np
    gc.collect()
    torch.cuda.empty_cache()

    return frames


# ── Pré-processamento da imagem ────────────────────────────────────────────────
print(f"Carregando imagem de entrada: {INPUT_IMAGE_PATH}")
input_image = preprocess_image(INPUT_IMAGE_PATH, MAX_AREA)
print(f"Dimensões após redimensionamento: {input_image.size[0]}×{input_image.size[1]} px")

# ── Geração do segmento inicial ────────────────────────────────────────────────
print("\n🎬 Gerando segmento inicial...")
current_frames = generate_segment(
    pipe,
    input_image,
    PROMPT,
    neg_prompt=NEG_PROMPT,
    seed=42,
    output_path=str(OUTPUT_DIR / "segment_000.mp4"),
)
all_segments = [(current_frames, OUTPUT_DIR / "segment_000.mp4")]
print(f"  ✅ Segmento 0: {len(current_frames)} frames → {OUTPUT_DIR/'segment_000.mp4'}")

# ── Loops de extensão ─────────────────────────────────────────────────────────
for i in range(1, EXTENSION_LOOPS + 1):
    print(f"\n🔁 Extensão {i}/{EXTENSION_LOOPS}...")
    last_frame = current_frames[-1].copy()   # último frame como nova imagem de entrada
    seg_path   = OUTPUT_DIR / f"segment_{i:03d}.mp4"
    current_frames = generate_segment(
        pipe,
        last_frame,
        PROMPT,
        neg_prompt=NEG_PROMPT,
        seed=42 + i,
        output_path=str(seg_path),
    )
    all_segments.append((current_frames, seg_path))
    print(f"  ✅ Segmento {i}: {len(current_frames)} frames → {seg_path}")

print("\n✅ Processamento concluído.")

## 7. Concatenação e Exibição

Os segmentos são concatenados removendo `OVERLAP_FRAMES` frames do início de cada extensão
(exceto o primeiro segmento) para evitar duplicação do frame de transição.

In [ ]:
def concatenate_segments(
    segments: list,
    overlap: int,
    output_path: Path,
    fps: int = FPS,
) -> Path:
    """
    Une todos os segmentos em um único vídeo, descartando `overlap` frames
    do início de cada segmento (exceto o primeiro) para remover duplicatas
    do frame de transição.
    """
    with imageio.get_writer(
        str(output_path), fps=fps, codec="libx264", quality=8, macro_block_size=1
    ) as writer:
        for idx, (frames, _) in enumerate(segments):
            start = 0 if idx == 0 else overlap
            for frame in frames[start:]:
                writer.append_data(np.array(frame))
    return output_path


final_path = concatenate_segments(
    all_segments,
    overlap=OVERLAP_FRAMES,
    output_path=OUTPUT_DIR / "final_video.mp4",
)

total_frames = sum(len(f) for f, _ in all_segments) - OVERLAP_FRAMES * EXTENSION_LOOPS
print(f"🎞️  Vídeo final: {total_frames} frames / {total_frames/FPS:.1f}s → {final_path}")
display(Video(str(final_path), embed=True, width=640))